# Open Drawer — 100 Demonstrations

In [ ]:
import numpy as np, matplotlib.pyplot as plt, json, random
from pathlib import Path
DATA = Path('/home/shareef/robot_zoo/data/demos/franka_open_drawer')
meta = json.load(open(DATA/'metadata.json'))
print('Task:', meta['task'], '| Episodes:', meta['total_episodes'], '| Success rate:', meta['success_rate'])

In [ ]:
episodes = sorted(DATA.glob('episode_*.npz'))
ep = np.load(episodes[0])
print('Keys:', list(ep.keys()))
print('observation_state:', ep['observation_state'].shape)
print('action:', ep['action'].shape)

In [ ]:
# Joint trajectories for 3 random episodes
sample = random.sample(episodes, 3)
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, path in zip(axes, sample):
    ep = np.load(path)
    jp = ep['observation_state'][:, :9]
    for j in range(7): ax.plot(jp[:, j], label=f'j{j}', alpha=0.8)
    ax.set_title(f'{path.name} | steps: {len(jp)}')
    ax.set_ylabel('Joint pos (rad)'); ax.legend(fontsize=7, ncol=4); ax.grid(alpha=0.3)
plt.suptitle('Open Drawer — Joint Trajectories', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'joint_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Reward (drawer opening distance) over time for one episode
ep = np.load(episodes[0])
reward = ep['reward']
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(reward, color='steelblue', linewidth=2)
ax.axhline(y=0.15, color='red', linestyle='--', label='Success threshold 0.15m')
ax.set_title('Drawer Opening Distance Over Time — Episode 0')
ax.set_xlabel('Timestep'); ax.set_ylabel('Distance (m)'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA/'drawer_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset statistics
lengths = [len(np.load(e)['reward']) for e in episodes]
max_rewards = [np.load(e)['reward'].max() for e in episodes]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lengths, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Episode Length Distribution'); axes[0].set_xlabel('Steps')
axes[1].hist(max_rewards, bins=20, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0.15, color='red', linestyle='--', label='Threshold')
axes[1].set_title('Max Drawer Opening per Episode'); axes[1].set_xlabel('Distance (m)'); axes[1].legend()
plt.suptitle('Open Drawer — Dataset Statistics', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'dataset_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Mean length: {np.mean(lengths):.1f} | Mean max opening: {np.mean(max_rewards):.3f}m')

In [ ]:
# Action distribution
all_actions = np.concatenate([np.load(e)['action'] for e in episodes])
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
labels = [f'arm_j{i}' for i in range(7)] + ['gripper']
for i, (ax, lbl) in enumerate(zip(axes.flatten(), labels)):
    ax.hist(all_actions[:, i], bins=30, alpha=0.7, color='purple' if i<7 else 'orange', edgecolor='black')
    ax.set_title(lbl); ax.grid(alpha=0.3)
plt.suptitle('Action Distribution — Open Drawer', fontsize=13)
plt.tight_layout()
plt.savefig(DATA/'action_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Total timesteps: {len(all_actions):,}')